# TCC Data Preparation
## Stage 1: Download, process, and store the pouring dataset

This notebook downloads the multiview pouring dataset from HuggingFace,
extracts frames from videos, and stores the processed data to the
configured storage backend (local disk or S3).

**Config:** Edit `configs/pouring.yaml` to switch between local and S3 storage.

**Usage:**
```bash
# Local execution
make data-prep

# With papermill parameters
make data-prep NB_PARAMS="-p config_path configs/pouring.yaml"
```

In [ ]:
import sys
import os
import pathlib
import subprocess

IN_COLAB = "google.colab" in sys.modules

print("Python:", sys.version)
print("Running in Colab:", IN_COLAB)

In [ ]:
if IN_COLAB:
    REPO_DIR = pathlib.Path("tcc")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/aegean-ai/tcc", str(REPO_DIR)], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "./tcc[notebooks]"], check=True)
else:
    print("Skipping clone/install \u2014 running inside dev container.")

In [ ]:
# \u2500\u2500 Papermill parameters \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
config_path = "configs/pouring.yaml"
fps = 15
image_width = 224
image_height = 224

In [ ]:
from tcc.storage import load_storage_config

storage = load_storage_config(config_path)

print(f"Storage backend: {storage.storage_backend}")
print(f"Dataset name:    {storage.dataset_name}")
print(f"Raw dir:         {storage.raw_dir}")
print(f"Processed dir:   {storage.processed_dir}")
if storage.cache_dir:
    print(f"Cache dir:       {storage.cache_dir}")

In [ ]:
if storage.storage_backend == "local":
    RAW_DIR = pathlib.Path(storage.raw_dir)
    PROCESSED_DIR = pathlib.Path(storage.processed_dir)
else:
    # S3 mode: use cache_dir for local staging
    RAW_DIR = pathlib.Path(storage.cache_dir) / "raw"
    PROCESSED_DIR = pathlib.Path(storage.cache_dir) / "processed" / storage.dataset_name

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"Local raw dir:       {RAW_DIR.resolve()}")
print(f"Local processed dir: {PROCESSED_DIR.resolve()}")

## 1. Download raw data from HuggingFace

The multiview pouring dataset is at
[`sermanet/multiview-pouring`](https://huggingface.co/datasets/sermanet/multiview-pouring).

In [ ]:
from huggingface_hub import snapshot_download

hf_cache_path = snapshot_download(
    repo_id="sermanet/multiview-pouring",
    repo_type="dataset",
    local_dir=str(RAW_DIR),
)

print("Dataset downloaded to:", hf_cache_path)

for split_dir in sorted(RAW_DIR.iterdir()):
    if split_dir.is_dir() and not split_dir.name.startswith("."):
        tfrecords = list(split_dir.glob("*.tfrecord*"))
        print(f"  {split_dir.name}/: {len(tfrecords)} TFRecord file(s)")

## 2. Convert videos to image-folder layout

Extract frames from `.mov` videos into the directory structure expected
by `tcc.datasets.VideoDataset`:

```
processed_dir/
\u2514\u2500\u2500 pouring/
    \u251c\u2500\u2500 train/video_001/frame_0000.png
    \u2514\u2500\u2500 val/video_050/frame_0000.png
```

In [ ]:
VIDEO_INPUT_DIR = RAW_DIR / "videos"
print(f"Converting videos from {VIDEO_INPUT_DIR}...")
print(f"Output: {PROCESSED_DIR}")

subprocess.run(
    [sys.executable, "-m", "tcc.dataset_preparation.videos_to_dataset",
     "--input-dir", str(VIDEO_INPUT_DIR),
     "--output-dir", str(PROCESSED_DIR.parent),
     "--name", storage.dataset_name,
     "--fps", str(fps),
     "--width", str(image_width),
     "--height", str(image_height),
     "--file-pattern", "**/*.mov",
     "--rotate"],
    check=True)

total = sum(len(f) for _, _, f in os.walk(PROCESSED_DIR))
print(f"Conversion complete: {total} files")

## 3. Upload to S3 (if S3 backend is configured)

When `storage_backend=s3`, upload the processed frames to the lakehouse
so that the training notebook can read directly from S3.

In [ ]:
if storage.storage_backend == "s3":
    import fsspec

    fs = fsspec.filesystem("s3", **storage.s3_storage_options)

    local_processed = PROCESSED_DIR
    s3_dest = storage.processed_dir

    print(f"Uploading {local_processed} -> {s3_dest}")
    fs.put(str(local_processed), s3_dest, recursive=True)

    n_files = len(fs.ls(s3_dest, detail=False))
    print(f"Upload complete: {n_files} top-level entries at {s3_dest}")
else:
    print("Local backend \u2014 no S3 upload needed.")
    print(f"Processed data ready at: {PROCESSED_DIR.resolve()}")

In [ ]:
print("\n=== Data preparation complete ===")
print(f"Backend:       {storage.storage_backend}")
print(f"Processed at:  {storage.processed_dir}")
print("\nRun the training notebook next:")
print("  make train")